# NestedKriging on the Branin 2D function (R)

This notebook demonstrates **NestedKriging**, a divide-and-conquer Gaussian Process for
large designs: the data are partitioned into $p$ groups, one Kriging submodel is fitted
per group (all sharing a **common prior** after hyperparameter unification), and
predictions are aggregated.

Available aggregations:
- **NK** (default): the optimal aggregation of Rullière, Durrande, Bachoc & Chevalier
  (*Statistics & Computing*, 2018) — itself a kriging predictor: it interpolates the
  design and provides consistent variances;
- **PoE / gPoE / BCM / rBCM**: precision-weighted products of experts (cheaper, but
  no interpolation guarantee).

Fit cost drops from $O(n^3)$ to $O(n^3/p^2)$ per likelihood evaluation.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 800$)
4. Fit `NestedKriging` and inspect the partition
5. Predict on a grid: mean, stdev, RMSE, interpolation check
6. Compare aggregations
7. Warped variant (WarpKriging submodels)


## 0. Installation (run once)

Build the C++ core and the R binding from source.
Requires: `cmake`, a C++ compiler, and R development headers.

The script `tools/r-linux-macos/build.sh` calls `tools/linux-macos/build.sh`
to build the C++ core, then runs `make` in `bindings/R` to compile and install
**rlibkriging** into `bindings/R/Rlibs`.

In [ ]:
# Run this cell once to build and install rlibkriging.
# Skip if already built (bindings/R/Rlibs/rlibkriging exists).
repo_root <- normalizePath(file.path(getwd(), "../.."), mustWork = FALSE)
rlibs     <- file.path(repo_root, "bindings", "R", "Rlibs", "rlibkriging")

if (!dir.exists(rlibs)) {
  message("Building rlibkriging from source…")
  ret <- system(paste0("cd '", repo_root, "' && bash tools/r-linux-macos/build.sh"))
  if (ret != 0) stop("Build failed — check compiler and cmake installation.")
} else {
  message("rlibkriging already built, skipping.")
}

## 1. Load rlibkriging

In [ ]:
repo_root <- normalizePath(file.path(getwd(), "../.."), mustWork = FALSE)
lib_path  <- file.path(repo_root, "bindings", "R", "Rlibs")
library(rlibkriging, lib.loc = lib_path)

## 2. Branin function

In [ ]:
branin <- function(X) {
  x1 <- X[, 1] * 15 - 5
  x2 <- X[, 2] * 15
  (x2 - 5 / (4 * pi^2) * x1^2 + 5 / pi * x1 - 6)^2 +
    10 * (1 - 1 / (8 * pi)) * cos(x1) + 10
}

grid_x <- seq(0, 1, length.out = 50)
grid_pts <- as.matrix(expand.grid(grid_x, grid_x))
z_true <- matrix(branin(grid_pts), 50, 50)
filled.contour(grid_x, grid_x, z_true, main = "True Branin")

## 3. Design of experiments ($n = 800$)

In [ ]:
set.seed(123)
n <- 800
X <- matrix(runif(2 * n), ncol = 2)
y <- branin(X)
dim(X)

## 4. Fit NestedKriging (NK aggregation, 8 groups)

In [ ]:
nk <- NestedKriging(y, X, kernel = "matern5_2", nb_groups = 8)
print(nk)

In [ ]:
groups <- nk$groups()
cols <- integer(nrow(X))
for (g in seq_along(groups)) cols[groups[[g]]] <- g
plot(X, col = cols, pch = 20, cex = 0.6, main = "k-means partition of the design",
     xlab = "x1", ylab = "x2")

## 5. Prediction on the grid

In [ ]:
p <- predict(nk, grid_pts)
filled.contour(grid_x, grid_x, matrix(p$mean, 50, 50), main = "NK mean")
filled.contour(grid_x, grid_x, matrix(p$stdev, 50, 50), main = "NK stdev")

cat("RMSE vs true Branin :", sqrt(mean((p$mean - as.vector(z_true))^2)), "\n")
pX <- predict(nk, X)
cat("max |mean - y| at design points (interpolation):", max(abs(pX$mean - y)), "\n")

## 6. Comparing aggregations

In [ ]:
for (agg in c("NK", "gPoE", "rBCM", "PoE", "BCM")) {
  k <- NestedKriging(y, X, kernel = "matern5_2", nb_groups = 8, aggregation = agg)
  pk <- predict(k, grid_pts)
  cat(sprintf("%-5s RMSE = %.4f\n", agg, sqrt(mean((pk$mean - as.vector(z_true))^2))))
}

## 7. Warped variant

In [ ]:
nkw <- NestedKriging(y, X, kernel = "gauss", nb_groups = 8,
                     warping = c("kumaraswamy", "kumaraswamy"))
pw <- predict(nkw, grid_pts)
cat("warping      :", paste(nkw$warping(), collapse = ", "), "\n")
cat("RMSE (warped):", sqrt(mean((pw$mean - as.vector(z_true))^2)), "\n")

## Notes

- The NK aggregation **interpolates** whatever the hyperparameters: it is a property of
  the aggregation itself, not of the fit.
- NK requires a **constant trend**; use the PoE family for other trends.
- Memory/speed of the NK prediction can be tuned with `set_predict_chunk`.
- In the warped case, the prior hyperparameters $(\theta, \text{warp})$ are estimated by a
  **single reference fit on a global subsample** (default 1000 points, see
  `set_warp_subsample`), then submodels are fitted in closed form (`optim="none"`).
